In [ ]:

import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/q3-ka-ai-2026/Q3_data.csv")# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(20)
#df.isnull().sum()

In [ ]:
empty_cols = ['D_87',
	"D_88",
	"B_39",	"D_110",
	"D_111",
	"D_108",
	"B_42",
  "D_73",	"D_135",
	"D_136",
	"D_138",
	"D_134",
	"D_137",
  "R_9",
	"B_29",
	"D_106",
	"D_132",
	"D_49",
	"D_66",
  "D_142"]

df = df.drop(empty_cols,axis=1)
df.fillna(0)

In [ ]:
# Task 2: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import MinMaxScaler

features = df.columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 5: Write your code here:
df['Target'].hist() # There is an imbalance

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
%pip install kagglehub catboost xgboost tqdm -q

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = []
pres = []
recall1= []
f11 = []
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training ")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred, zero_division=0)
  recall = recall_score(y_test, y_pred, zero_division=0)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  acc.append(accuracy)
  pres.append(precision)
  recall1.append(recall)
  f11.append(f1)

In [ ]:
import numpy as np
print(f"  Accuracy:  {np.mean(acc):.4f}")
print(f"  Precision: {np.mean(pres):.4f}")
print(f"  Recall:    {np.mean(recall1):.4f}")
print(f"  F1-Score:  {np.mean(f11):.4f}")

In [ ]:
import matplotlib.pyplot as plt
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(100,100))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# As the grapgh shows the moste important feature is P_2


In [ ]:
# Task Bonus: Write your code here:
X = df['P_2']
y = df['Target']
acc = []
pres = []
recall1= []
f11 = []
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training ")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  # 3. Save metrics for that model in this fold
  accuracy = (y_pred,y_test)
  precision = precision_score(y_test, y_pred, zero_division=0)
  recall = recall_score(y_test, y_pred, zero_division=0)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  acc.append(accuracy)
  pres.append(precision)
  recall1.append(recall)
  f11.append(f1)